# 第 8 章の実習　標本抽出と偏り

全体の体重の平均を知るために，一回に選ぶ件数を増やした場合と，選ぶ範囲を替えた場合とでは，何が変わるでしょうか。全 342 件から各 20 件を選ぶ実験を基準にして，二つの条件のうち一つだけを変えます。

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "penguins.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

penguins = pd.read_csv(csv_path)
has_measurements = (
    penguins["body_mass_g"].notna()
    & penguins["bill_length_mm"].notna()
    & penguins["bill_depth_mm"].notna()
    & penguins["flipper_length_mm"].notna()
)
population = penguins.loc[has_measurements]
target_mean = population["body_mass_g"].mean()
print(len(population), round(target_mean, 2))

## 1. 基準を読み，予想する（約 5 分）

基準は，第 7 章で使ったペンギンの測定記録 342 件の平均体重，約 4,201.75 g です。次のコードは，全 342 件から各 20 件を非復元で選び，平均を一つずつ残す実験を 2,000 回行います。

図の横軸は一回の平均（g），縦軸はその区間に入った平均の個数です。破線は全 342 件の平均，淡い帯はそこから 200 g 以内の範囲です。

In [ ]:
B = 2000
baseline_means = []

for seed in range(B):
    sample = population.sample(n=20, replace=False, random_state=seed)
    baseline_means.append(sample["body_mass_g"].mean())

baseline_close = (pd.Series(baseline_means) - target_mean).abs() <= 200
print(baseline_close.sum(), len(baseline_means))

bins = range(3400, 5401, 50)
fig, ax = plt.subplots(layout="constrained")
ax.axvspan(target_mean - 200, target_mean + 200, color="#e4e5db", zorder=0)
ax.hist(baseline_means, bins=bins, color="#286b70", edgecolor="white")
ax.axvline(target_mean, color="#4a4c56", linestyle="--")
ax.set(title="All islands: 20 per sample", xlabel="Mean body mass (g)",
       ylabel="Samples", xlim=(3400, 5400), ylim=(0, 650))
plt.show()

A と B のうち一つを選びます。予想を書くときは，図の中心がどこに来るかと，平均の散らばり方が基準の図とどう違いそうかを考えます。

| 選択 | 変えるもの | そのまま使う条件 |
|---|---|---|
| A | 一回に選ぶ件数を 40 にする | 全 342 件から選び，2,000 回繰り返す |
| B | Dream の 124 件から選ぶ | 一回に 20 件を選び，2,000 回繰り返す |

B を考える手掛かりとして，Dream の表には Adelie が 56 件，Chinstrap が 68 件あります。Gentoo は含まれません。前章で見たペンギンの種類別の体重分布を思い出して，どこに平均が集まりそうかを考えます。

（選択と予想をここに記入）

## 2. 条件を一つ替える（約 9 分）

A なら `n` を 40 にする一行を書きます。B なら `island` が `"Dream"` の行を条件で選び，その表を `frame` に入れる一行を書きます。もう一方の条件はそのまま使います。

次に，繰り返しの中で `sample_mean` を `means` に追加する一行を書きます。一回ごとに 20 件や 40 件の表が得られますが，ここで集めるのは各回の平均一つずつです。`append()` の行を，平均を計算した行と同じ字下げにします。

条件と保存の行を記入したらセルを実行し，左に抽出枠の件数，中央に一回に選ぶ件数，右に保存した平均の件数が出ることを確かめます。未記入のまま実行すると，平均の保存件数は 0 のままです。

In [ ]:
frame = population
n = 20
# ここに A または B の条件を設定する一行を追加する。

means = []
for seed in range(B):
    sample = frame.sample(n=n, replace=False, random_state=seed)
    sample_mean = sample["body_mass_g"].mean()
    # ここに今回の平均を保存する一行を追加する。

print(len(frame), n, len(means))

基準の図と描き直した図を，同じ区間・同じ軸で並べます。両方の破線は全 342 件の平均です。図では平均との差が 200 g 以内の範囲を探し，出力ではその範囲に入った回数を確かめます。

In [ ]:
if len(means) != B:
    print("平均を 2,000 個保存してから，このセルをもう一度実行します。")
elif n == 20 and len(frame) == len(population):
    print("A または B の条件を設定してから，比較図を作ります。")
else:
    change_close = (pd.Series(means) - target_mean).abs() <= 200
    print(change_close.sum(), len(means))

    fig, (base_ax, changed_ax) = plt.subplots(
        2, 1, sharex=True, sharey=True, layout="constrained"
    )
    base_ax.axvspan(target_mean - 200, target_mean + 200, color="#e4e5db", zorder=0)
    base_ax.hist(baseline_means, bins=bins, color="#286b70", edgecolor="white")
    base_ax.axvline(target_mean, color="#4a4c56", linestyle="--")
    base_ax.set(title="All islands: 20 per sample", ylabel="Samples")

    changed_ax.axvspan(target_mean - 200, target_mean + 200, color="#e4e5db", zorder=0)
    changed_ax.hist(means, bins=bins, color="#b15334", edgecolor="white")
    changed_ax.axvline(target_mean, color="#4a4c56", linestyle="--")
    changed_ax.set(title=f"Frame: {len(frame)} records, {n} per sample",
                   xlabel="Mean body mass (g)", ylabel="Samples",
                   xlim=(3400, 5400), ylim=(0, 650))
    plt.show()

<details>
<summary>条件と保存の書き方を確認する</summary>

A を選んだら，`n = 20` のあとに `n = 40` を書きます。B を選んだら，条件を指定する行に次を加えます。

```python
frame = population.loc[population["island"] == "Dream"]
```

どちらの選択でも，`sample_mean` を計算した直後に，次の行を同じ字下げで書きます。

```python
    means.append(sample_mean)
```

</details>

## 3. 何が変わったか説明する（約 6 分）

最初の予想を見返し，図と回数を使って 2〜3 文で説明します。

1. 一回の件数と抽出枠のうち，何を変え，何を保ったか。
2. 平均がどこに集まり，幅や基準線との距離はどう変わったか。
3. 全 342 件の平均から 200 g 以内に入る回数はどう変わったか。

（ここに記入）

提出前にカーネルを再起動し，上から全セルを実行します。基準と自分の条件の図，保存した平均の件数，記入した説明がそろったノートブックを保存します。

### 出典

2007〜2009 年に南極半島付近の 3 島で調査された成鳥の測定記録を使います。Kristen Gorman と Palmer Station LTER の研究者による調査から，Horst，Hill，Gorman が整理・配布した表です。今回の抽出はその表を使った計算実験です。[データの配布元](https://allisonhorst.github.io/palmerpenguins/) ・ [原著](https://doi.org/10.1371/journal.pone.0090081)